# Módulo 03: Álgebra Matricial, Menores, Cofactores e Inversión de Matrices

> **Institución:** Universidad San Sebastián (USS) — Sede Patagonia  
> **Carrera:** Ingeniería Civil Informática  
> **Asignatura:** Álgebra Lineal  
> **Docente:** Carol Asencio González  
> **Entorno:** Python 3, SymPy, NumPy, Matplotlib, ipywidgets  

---

### Trazabilidad de Fuentes
[Cátedra USS — Diapositivas Docente] | [Texto Guía — Stanley Grossman] | [Computación Científica]

- **[Cátedra USS — Diapositivas Docente]:** Teorema de Laplace, cálculo de menores $M_{ij}$, matriz de cofactores $\operatorname{Cof}(A)$, transpuesta adjunta $\operatorname{Adj}(A) = [\operatorname{Cof}(A)]^T$ y fórmula analítica $A^{-1} = \frac{1}{\det A}\operatorname{Adj}(A)$.
- **[Texto Guía — Stanley Grossman]:** Demostración del Teorema Fundamental $A \cdot \operatorname{Adj}(A) = \det(A) \cdot I$, regularidad de matrices e inversión analítica.
- **[Computación Científica]:** Aritmética fraccionaria exacta con SymPy y diagramas matriciales térmicos comparativos en Matplotlib.

---

> [!NOTE]
> La matriz inversa $A^{-1}$ es fundamental para la resolución analítica de sistemas de la forma $A \mathbf{x} = \mathbf{b} \implies \mathbf{x} = A^{-1} \mathbf{b}$. El método de la matriz adjunta proporciona una fórmula cerrada para construir la inversa exacta sin recurrir a operaciones elementales por filas, resultando especialmente idóneo en matrices con coeficientes paramétricos o simbólicos.

## 1. Fundamentos Teóricos

### 1.1 Menores Complementarios y Matriz de Cofactores
Dada una matriz cuadrada $A = (a_{ij}) \in \mathcal{M}_n(\mathbb{R})$:

1. **Menor Complementario ($M_{ij}$):** Determinante de la submatriz de orden $(n-1) \times (n-1)$ obtenida al suprimir la fila $i$ y la columna $j$ de $A$.
2. **Cofactor ($C_{ij}$):** Menor ponderado por el factor de posición ajedrezado:
   $$C_{ij} = (-1)^{i+j} M_{ij}$$
3. **Tablero de Signos:** Alternancia posicional en dimensión $3 \times 3$:
   $$S = \begin{pmatrix} + & - & + \\ - & + & - \\ + & - & + \end{pmatrix}$$

---

### 1.2 Teorema de Laplace (Expansión por Filas o Columnas)
El determinante de $A$ puede evaluarse fijando cualquier fila $i$ o columna $j$:

$$
\det(A) = \sum_{j=1}^n a_{ij} C_{ij} = \sum_{j=1}^n a_{ij} (-1)^{i+j} M_{ij} \quad (\text{por la fila } i)
$$

> [!TIP]
> Para minimizar operaciones algebraicas en certámenes, seleccionar siempre la fila o columna que posea la mayor cantidad de ceros.

---

### 1.3 Matriz Adjunta Clásica y Teorema Fundamental

> [!IMPORTANT]
> **Definición de Matriz Adjunta:**  
> La **matriz adjunta** $\operatorname{Adj}(A)$ se define estrictamente como la **transpuesta de la matriz de cofactores**:
> $$\operatorname{Adj}(A) = [\operatorname{Cof}(A)]^T$$
>
> Para toda matriz cuadrada $A \in \mathcal{M}_n(\mathbb{R})$ se verifica la identidad fundamental:
> $$A \cdot \operatorname{Adj}(A) = \operatorname{Adj}(A) \cdot A = \det(A) \cdot I_n$$
>
> Si $\det(A) \neq 0$, la matriz es invertible y su inversa exacta viene dada por:
> $$\boxed{\,A^{-1} = \frac{1}{\det(A)} \operatorname{Adj}(A) = \frac{1}{\det(A)} [\operatorname{Cof}(A)]^T\,}$$

## 2. Implementación Computacional

In [ ]:
# Importaciones y configuración de visualización
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Paleta USS
USS_BLUE = '#00205B'
USS_GOLD = '#D4AF37'
USS_ACCENT_BLUE = '#1E88E5'
USS_ACCENT_GREEN = '#27AE60'
USS_ACCENT_RED = '#C0392B'
USS_DARK_GRAY = '#2C3E50'
USS_LIGHT_GRAY = '#F8F9F9'

sp.init_printing(use_latex='mathjax')
%matplotlib inline
print('Entorno de álgebra matricial USS inicializado.')

In [ ]:
class InversorCofactores:
    """Calcula paso a paso menores, cofactores, adjunta e inversa exacta."""
    def __init__(self, data):
        self.A_sp = sp.Matrix(data)
        self.n = self.A_sp.rows
        self.det_A = self.A_sp.det()
        self.es_invertible = (self.det_A != 0)
        
    def menor_submatriz(self, i, j):
        filas = [r for r in range(self.n) if r != i]
        cols = [c for c in range(self.n) if c != j]
        return self.A_sp.extract(filas, cols)
        
    def matriz_menores_y_cofactores(self):
        M = sp.zeros(self.n, self.n)
        S = sp.zeros(self.n, self.n)
        C = sp.zeros(self.n, self.n)
        
        for i in range(self.n):
            for j in range(self.n):
                signo = (-1)**(i + j)
                sub_M = self.menor_submatriz(i, j)
                det_sub = sub_M.det()
                
                M[i, j] = det_sub
                S[i, j] = signo
                C[i, j] = signo * det_sub
                
        return M, S, C
        
    def matriz_adjunta(self, C):
        return C.T
        
    def inversa(self):
        M, S, C = self.matriz_menores_y_cofactores()
        Adj = self.matriz_adjunta(C)
        if not self.es_invertible:
            return None, M, S, C, Adj, None
            
        inv_A = (sp.Rational(1, self.det_A)) * Adj
        verif = self.A_sp * inv_A
        return inv_A, M, S, C, Adj, verif

## 3. Visualización y Verificación Matricial

In [ ]:
def graficar_tableros_matriciales(inversor):
    """Genera 6 tableros visuales con mapas de calor de alto contraste USS."""
    inv_A, M, S, C, Adj, verif = inversor.inversa()
    n = inversor.n
    
    fig, axes = plt.subplots(2, 3, figsize=(14, 9), facecolor='white')
    fig.suptitle(
        f"Descomposición por Cofactores e Inversa Matricial\n"
        f"Orden: {n}x{n} | det(A) = {inversor.det_A} | "
        f"{'Invertible (No Singular)' if inversor.es_invertible else 'Singular (No Invertible)'}",
        fontsize=13, fontweight='bold', color=USS_BLUE, y=0.98
    )
    
    titulos = [
        "1. Matriz Original $A$",
        "2. Tablero de Signos $(-1)^{i+j}$",
        r"3. Matriz de Cofactores $\operatorname{Cof}(A)$",
        r"4. Matriz Adjunta $\operatorname{Adj}(A) = [\operatorname{Cof}(A)]^T$",
        r"5. Matriz Inversa $A^{-1} = \frac{1}{\det A}\operatorname{Adj}(A)$",
        r"6. Verificación $A \cdot A^{-1} = I$"
    ]
    
    mats = [
        inversor.A_sp,
        S,
        C,
        Adj,
        inv_A if inv_A is not None else sp.zeros(n, n),
        verif if verif is not None else sp.zeros(n, n)
    ]
    
    cmap_blue = mcolors.LinearSegmentedColormap.from_list('uss_b', [USS_LIGHT_GRAY, USS_ACCENT_BLUE, USS_BLUE])
    cmap_gold = mcolors.LinearSegmentedColormap.from_list('uss_g', [USS_LIGHT_GRAY, USS_GOLD, '#B7950B'])
    cmap_green = mcolors.LinearSegmentedColormap.from_list('uss_gr', [USS_LIGHT_GRAY, USS_ACCENT_GREEN, '#1E8449'])
    cmaps = [cmap_blue, cmap_gold, cmap_blue, cmap_gold, cmap_blue, cmap_green]
    
    for idx, ax in enumerate(axes.flat):
        m = mats[idx]
        m_num = np.array(m.tolist(), dtype=float)
        max_v = max(1.0, float(np.max(np.abs(m_num))))
        
        ax.imshow(m_num, cmap=cmaps[idx], vmin=-max_v, vmax=max_v, aspect='auto')
        ax.set_title(titulos[idx], fontsize=11, fontweight='bold', color=USS_BLUE, pad=8)
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels([f"C{j+1}" for j in range(n)], fontsize=9, color=USS_DARK_GRAY)
        ax.set_yticklabels([f"F{i+1}" for i in range(n)], fontsize=9, color=USS_DARK_GRAY)
        
        for i in range(n):
            for j in range(n):
                val_sp = m[i, j]
                if idx == 1:
                    val_str = "+" if val_sp == 1 else "-"
                elif isinstance(val_sp, sp.Rational) and val_sp.q != 1:
                    val_str = f"{val_sp.p}/{val_sp.q}"
                else:
                    val_str = f"{val_sp}"
                    
                c_val = m_num[i, j]
                txt_col = 'white' if abs(c_val) > 0.5 * max_v else USS_DARK_GRAY
                ax.text(j, i, val_str, ha="center", va="center", color=txt_col,
                        fontweight='bold', fontsize=12)
                        
        for spine in ax.spines.values():
            spine.set_color(USS_DARK_GRAY)
            spine.set_linewidth(1.1)
            
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()

In [ ]:
# Selector interactivo de matrices de estudio
ejemplos_matrices = {
    '1. Matriz Clásica Invertible 3x3': [
        [1, 2, -1],
        [2, 0, 1],
        [1, 1, 1]
    ],
    '2. Matriz Simétrica Invertible': [
        [2, -1, 0],
        [-1, 2, -1],
        [0, -1, 2]
    ],
    '3. Matriz Triangular Superior': [
        [3, 4, 1],
        [0, 2, 5],
        [0, 0, -1]
    ],
    '4. Matriz Singular (det = 0, F3 = F1 + F2)': [
        [1, 2, 3],
        [4, 5, 6],
        [5, 7, 9]
    ]
}

dropdown_mat = widgets.Dropdown(
    options=list(ejemplos_matrices.keys()),
    value=list(ejemplos_matrices.keys())[0],
    description='Matriz A:',
    layout=widgets.Layout(width='65%')
)

out_mat_sim = widgets.Output()

def actualizar_matriz(change=None):
    with out_mat_sim:
        clear_output(wait=True)
        data = ejemplos_matrices[dropdown_mat.value]
        inv_calc = InversorCofactores(data)
        
        estado_txt = 'La matriz es invertible. Posee inversa única.' if inv_calc.es_invertible else 'La matriz es singular. No existe inversa.'
        b_color = USS_ACCENT_GREEN if inv_calc.es_invertible else USS_ACCENT_RED
        
        display(HTML(f"""
        <div style="background-color: {USS_LIGHT_GRAY}; padding: 10px 14px; border-left: 5px solid {b_color}; border-radius: 4px; margin-bottom: 12px;">
            <h4 style="margin: 0; color: {USS_BLUE};">Determinante: det(A) = {inv_calc.det_A}</h4>
            <p style="margin: 4px 0 0 0; color: {USS_DARK_GRAY}; font-size: 13px;">
                {estado_txt}
            </p>
        </div>
        """))
        
        graficar_tableros_matriciales(inv_calc)

dropdown_mat.observe(actualizar_matriz, names='value')
display(widgets.VBox([dropdown_mat, out_mat_sim]))
actualizar_matriz()

---

## Resumen de Fórmulas Clave para Evaluaciones

| Concepto | Fórmula Matemática | Propiedad Destacada |
|:---|:---|:---|
| **Menor** | $M_{ij} = \det(\text{Submatriz sin fila } i \text{ ni col } j)$ | Escalar de orden $(n-1) \times (n-1)$ |
| **Cofactor** | $C_{ij} = (-1)^{i+j} M_{ij}$ | Signo de posición ajedrezado |
| **Matriz Adjunta** | $\operatorname{Adj}(A) = [\operatorname{Cof}(A)]^T$ | Transpuesta: la fila $i$ de cofactores pasa a columna $i$ |
| **Inversa** | $A^{-1} = \frac{1}{\det A} \operatorname{Adj}(A)$ | Válida si y solo si $\det(A) \neq 0$ |
| **Determinante de la Adjunta** | $\det(\operatorname{Adj} A) = (\det A)^{n-1}$ | Identidad clásica de certámenes |